# Third-review GPU run 5 on Kaggle: fine-tuning pilot rerun (M2)

Reruns only `correction/finetune_byt5.py` (see `context/decisions.md` D50): the review3d run fed the model line breaks it
had never seen in training, so it skipped and repeated lines. Now the lines of each chunk are joined by spaces, as in
training, and gradients are accumulated so every update sees 8 pairs. About 3 h on one T4.

**Before running:**
1. Session options: **Accelerator = GPU T4 x2** (one GPU is used), Internet = on.
2. Add data: `reid2019-eval40` (holds `ocr_eval40.jsonl`) and `postocr-dev10` (holds `ocr_dev.jsonl`).
3. Add-ons, Secrets: nothing needed (ByT5 and Wikipedia are not gated).
4. Save Version, then "Save & Run All (Commit)". It runs in the background; the browser can be closed.

When it finishes, download `review3e.zip` from the version's Output tab.

In [ ]:
import glob, os, shutil, subprocess, sys, time

REPO_DIR = "/tmp/bengali-postocr-sllm"
OUT_DIR = "/kaggle/working/review3e"
os.makedirs(OUT_DIR, exist_ok=True)
SETUP_LOG = f"{OUT_DIR}/setup.log"

def sh(cmd):
    """Run a shell command, append its output to setup.log, return True on success."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    with open(SETUP_LOG, "a") as f:
        f.write(f"\n$ {cmd}\nexit {r.returncode}\n{r.stdout[-3000:]}{r.stderr[-3000:]}")
    print(("ok   " if r.returncode == 0 else "FAIL ") + cmd)
    return r.returncode == 0

if not os.path.isdir(REPO_DIR):
    sh(f"git clone -q https://github.com/Ariffurrhmn/bengali-postocr-sllm.git {REPO_DIR}")
assert os.path.isdir(REPO_DIR), "git clone failed: is Internet on in Session options? See review3e/setup.log"
# transformers pinned to the version of the main runs (10-12 October 2026); if that
# version will not install on this image, fall back to the latest and record it.
if not sh("pip install -q transformers==5.19.0 accelerate sentencepiece jiwer protobuf"):
    sh("pip install -q -U transformers accelerate sentencepiece jiwer protobuf")
sh('pip install -q datasets "huggingface-hub<2"')

commit = subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout
import torch, transformers
env = (f"repo commit: {commit}python: {sys.version.split()[0]}\ntorch: {torch.__version__}\n"
       f"cuda: {torch.version.cuda}\ncudnn: {torch.backends.cudnn.version()}\n"
       f"transformers: {transformers.__version__}\ngpus: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}\n")
open(f"{OUT_DIR}/environment.txt", "w").write(env)
print(env)
assert torch.cuda.device_count() >= 1, "Set Accelerator to GPU T4 x2"


In [ ]:
# Gemma and Llama are gated. Without the token only their jobs fail; the rest still run.
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Hugging Face login ok")
except Exception as e:
    print("WARNING: no Hugging Face login (Gemma and Llama jobs will fail):", e)


In [ ]:
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"Attach the dataset that contains {name}"
    print(name, "->", hits[0])
    return hits[0]

OCR, DEV = find("ocr_eval40.jsonl"), find("ocr_dev.jsonl")


In [ ]:
# One job on GPU 0; progress in review3e/lane0.log (the last lines are printed when it ends).
with open(f"{OUT_DIR}/lane0.log", "a") as log:
    r = subprocess.run([sys.executable, "finetune_byt5.py", "--ocr-path", OCR, "--dev-path", DEV,
                        "--out-dir", OUT_DIR, "--train-minutes", "150"],
                       cwd=f"{REPO_DIR}/correction", env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"},
                       stdout=log, stderr=log)
print("exit", r.returncode)
print("".join(open(f"{OUT_DIR}/lane0.log").readlines()[-15:]))

In [ ]:
import collections, json
for f in sorted(glob.glob(f"{OUT_DIR}/*.jsonl")):
    rows = [json.loads(l) for l in open(f, encoding="utf-8") if l.strip()]
    errs = sum(1 for r in rows if r.get("error"))
    keys = collections.Counter((r.get("model_key") or r.get("model"), r.get("engine")) for r in rows)
    print(os.path.basename(f), len(rows), "rows", f"errors {errs}" if errs else "", dict(keys))
shutil.rmtree(f"{OUT_DIR}/byt5_best", ignore_errors=True)
shutil.make_archive("/kaggle/working/review3e", "zip", "/kaggle/working", "review3e")
print("wrote /kaggle/working/review3e.zip")
